# Spotify Playlist Data - Extended EDA (Part 2)

**Author:** Mike Stanton  
**Date:** November 22, 2025  
**Purpose:** Extended exploratory data analysis building on initial findings from mike_eda.ipynb. Focus on deeper analysis and advanced visualizations.

## Overview
This notebook continues the analysis from Part 1, incorporating:
- Advanced network analysis techniques
- User behavior patterns
- Recommendation system preparation
- Integration with team datasets

## Prerequisites
- Run `mike_eda.ipynb` first for initial data loading and cleaning
- Ensure all required packages are installed from `requirements.txt`

## 1. Import Libraries and Load Data

In [2]:
# Core data science libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Network analysis
import networkx as nx

# Data loading and processing
import zipfile
import os
import kagglehub
from collections import defaultdict, Counter
import itertools

# Machine learning and statistics
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy import stats

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

Libraries imported successfully!
pandas version: 2.2.2
numpy version: 1.26.4


## 2. Data Loading and Preparation

Load the cleaned dataset (assuming preprocessing from mike_eda.ipynb)

In [6]:
# Load the Spotify dataset with error handling
def load_spotify_data():
    """Load and clean the Spotify dataset"""
    try:
        # Download dataset
        dataset_path = kagglehub.dataset_download("andrewmvd/spotify-playlists")
        print(f"Dataset downloaded to: {dataset_path}")
        
        # Find CSV or ZIP files
        csv_files = []
        zip_files = []
        
        for file in os.listdir(dataset_path):
            file_path = os.path.join(dataset_path, file)
            if file.lower().endswith('.csv'):
                csv_files.append(file_path)
            elif file.lower().endswith('.zip'):
                zip_files.append(file_path)
        
        # Load data with robust parameters
        df = None
        
        if csv_files:
            csv_file = csv_files[0]
            print(f"Loading CSV file: {csv_file}")
            df = pd.read_csv(
                csv_file,
                encoding='iso-8859-1',
                sep=',',
                quotechar='"',
                escapechar='\\',
                engine='python',
                on_bad_lines='skip'
            )
        
        elif zip_files:
            zip_file = zip_files[0]
            print(f"Extracting from ZIP: {zip_file}")
            with zipfile.ZipFile(zip_file, 'r') as z:
                csv_name = next((n for n in z.namelist() if n.lower().endswith('.csv')), None)
                if csv_name:
                    with z.open(csv_name) as f:
                        df = pd.read_csv(
                            f,
                            encoding='iso-8859-1',
                            sep=',',
                            quotechar='"',
                            escapechar='\\',
                            engine='python',
                            on_bad_lines='skip'
                        )
        
        if df is not None:
            # Clean column names
            df.columns = [col.strip().strip('"') for col in df.columns]
            print(f"Dataset loaded successfully! Shape: {df.shape}")
            print(f"Columns: {list(df.columns)}")
            return df
        else:
            print("Failed to load dataset")
            return None
            
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

# Load the data
df = load_spotify_data()

Dataset downloaded to: C:\Users\mstan\.cache\kagglehub\datasets\andrewmvd\spotify-playlists\versions\1
Loading CSV file: C:\Users\mstan\.cache\kagglehub\datasets\andrewmvd\spotify-playlists\versions\1\spotify_dataset.csv
Dataset loaded successfully! Shape: (12791243, 4)
Columns: ['user_id', 'artistname', 'trackname', 'playlistname']
Dataset loaded successfully! Shape: (12791243, 4)
Columns: ['user_id', 'artistname', 'trackname', 'playlistname']


## 3. Advanced Data Analysis Framework

Set up analysis framework for deeper insights

In [7]:
# Verify data is loaded and show basic info
if df is not None:
    print("=== Dataset Overview ===")
    print(f"Shape: {df.shape}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print("\n=== Basic Statistics ===")
    print(f"Unique users: {df['user_id'].nunique():,}")
    print(f"Unique artists: {df['artistname'].nunique():,}")
    print(f"Unique tracks: {df['trackname'].nunique():,}")
    print(f"Unique playlists: {df['playlistname'].nunique():,}")
    
    print("\n=== Sample Data ===")
    display(df.head(3))
    
    print("\n=== Ready for advanced analysis! ===")
else:
    print("⚠️  Data not loaded. Please check the data loading section above.")

=== Dataset Overview ===
Shape: (12791243, 4)
Memory usage: 3390.16 MB

=== Basic Statistics ===
Unique users: 15,910
Unique artists: 287,438
Unique users: 15,910
Unique artists: 287,438
Unique tracks: 1,999,879
Unique playlists: 156,884

=== Sample Data ===
Unique tracks: 1,999,879
Unique playlists: 156,884

=== Sample Data ===


,user_id,artistname,trackname,playlistname
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010



=== Ready for advanced analysis! ===


## 3.5. Load Additional Music Metadata Dataset

Import the 900k Spotify dataset with audio features to enhance our analysis capabilities

In [4]:
# Load the full metadata dataset with robust error handling
def load_full_metadata(max_retries=3):
    """Load the complete Spotify metadata dataset with timeout handling"""
    import time
    
    for attempt in range(max_retries):
        try:
            print(f"Downloading full Spotify metadata dataset... (Attempt {attempt + 1}/{max_retries})")
            print("⚠️  This may take several minutes for the first download...")
            
            # Download with retry logic
            metadata_path = kagglehub.dataset_download("devdope/900k-spotify")
            print(f"✅ Dataset downloaded/cached at: {metadata_path}")
            
            # Find CSV files in the metadata dataset
            metadata_files = []
            for file in os.listdir(metadata_path):
                file_path = os.path.join(metadata_path, file)
                if file.lower().endswith('.csv'):
                    metadata_files.append(file_path)
                    size_mb = os.path.getsize(file_path) / (1024 * 1024)
                    print(f"Found metadata file: {file} ({size_mb:.1f} MB)")
            
            if not metadata_files:
                print("❌ No CSV files found in metadata dataset")
                return None
                
            # Load the main metadata file (usually the largest one)
            metadata_file = max(metadata_files, key=os.path.getsize)
            file_size_mb = os.path.getsize(metadata_file) / (1024 * 1024)
            print(f"Loading full dataset from: {os.path.basename(metadata_file)} ({file_size_mb:.1f} MB)")
            
            # Load the complete dataset with optimized settings
            print("📊 Reading full dataset... (this may take a few minutes)")
            df_metadata = pd.read_csv(
                metadata_file,
                encoding='utf-8',
                on_bad_lines='skip',
                low_memory=False,
                dtype_backend='pyarrow'  # Use faster backend if available
            )
            
            print(f"✅ Full metadata loaded successfully!")
            print(f"   Shape: {df_metadata.shape}")
            print(f"   Memory: {df_metadata.memory_usage(deep=True).sum() / (1024**2):.1f} MB")
            print(f"   Columns: {len(df_metadata.columns)}")
            
            return df_metadata
            
        except Exception as e:
            print(f"❌ Attempt {attempt + 1} failed: {e}")
            
            if "timed out" in str(e).lower() or "timeout" in str(e).lower():
                if attempt < max_retries - 1:
                    wait_time = (attempt + 1) * 15  # Progressive backoff
                    print(f"⏱️  Network timeout. Waiting {wait_time} seconds before retry...")
                    time.sleep(wait_time)
                    continue
                else:
                    print("❌ All download attempts failed due to network timeout.")
                    print("💡 Suggestions:")
                    print("   1. Check your internet connection")
                    print("   2. Try again during off-peak hours")
                    print("   3. Use a VPN if corporate firewall is blocking")
                    print("   4. Consider using the sample version temporarily")
                    return None
            else:
                print(f"❌ Non-timeout error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
                    continue
                else:
                    return None
    
    return None

# Load the full metadata dataset
print("=== Loading FULL Spotify Audio Features Dataset (900k rows) ===")
print("🔄 This will use cached version if already downloaded, or download fresh copy...")

df_metadata_full = load_full_metadata()

if df_metadata_full is not None:
    print("\n🎉 SUCCESS! Full metadata dataset loaded!")
    print(f"📈 Dataset contains {len(df_metadata_full):,} songs with {len(df_metadata_full.columns)} features")
    print("🚀 Ready for advanced content-based analysis!")
else:
    print("\n❌ Failed to load full dataset.")
    print("💡 You can still use the sample version or try again later.")

=== Loading FULL Spotify Audio Features Dataset (900k rows) ===
🔄 This will use cached version if already downloaded, or download fresh copy...
⚠️  This may take several minutes for the first download...
✅ Dataset downloaded/cached at: C:\Users\mstan\.cache\kagglehub\datasets\devdope\900k-spotify\versions\3
Found metadata file: spotify_dataset.csv (1095.9 MB)
Loading full dataset from: spotify_dataset.csv (1095.9 MB)
📊 Reading full dataset... (this may take a few minutes)
✅ Dataset downloaded/cached at: C:\Users\mstan\.cache\kagglehub\datasets\devdope\900k-spotify\versions\3
Found metadata file: spotify_dataset.csv (1095.9 MB)
Loading full dataset from: spotify_dataset.csv (1095.9 MB)
📊 Reading full dataset... (this may take a few minutes)
✅ Full metadata loaded successfully!
   Shape: (551443, 39)
   Memory: 1157.8 MB
   Columns: 39

🎉 SUCCESS! Full metadata dataset loaded!
📈 Dataset contains 551,443 songs with 39 features
🚀 Ready for advanced content-based analysis!
✅ Full metadata l

In [8]:
# Explore the metadata dataset structure
if df_metadata_full is not None:
    print("=== Metadata Dataset Overview ===")
    print(f"Shape: {df_metadata_full.shape}")
    print(f"Memory usage: {df_metadata_full.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print("\n=== Column Information ===")
    for i, col in enumerate(df_metadata_full.columns):
        non_null = df_metadata_full[col].notna().sum()
        dtype = str(df_metadata_full[col].dtype)
        print(f"{i+1:2d}. {col:20s} | {dtype:10s} | {non_null:,} non-null")
    
    print("\n=== Sample Metadata ===")
    display(df_metadata_full.head(3))
    
    # Check for potential matching columns with main dataset
    print("\n=== Potential Matching Columns ===")
    main_cols = set(df.columns) if df is not None else set()
    meta_cols = set(df_metadata_full.columns)
    common_cols = main_cols.intersection(meta_cols)
    
    if common_cols:
        print(f"Common columns: {list(common_cols)}")
    else:
        print("No exact column matches found.")
        print("Main dataset columns:", list(main_cols) if main_cols else "Not loaded")
        print("Metadata columns with 'name' or 'artist':", 
              [col for col in meta_cols if any(word in col.lower() for word in ['name', 'artist', 'track', 'song'])])
        
else:
    print("⚠️ Metadata not loaded. Please check the loading section above.")

=== Metadata Dataset Overview ===
Shape: (551443, 39)
Memory usage: 1157.80 MB

=== Column Information ===
 1. Artist(s)            | string[pyarrow] | 551,443 non-null
 2. song                 | string[pyarrow] | 551,427 non-null
 3. text                 | string[pyarrow] | 551,443 non-null
 4. Length               | string[pyarrow] | 551,443 non-null
 5. emotion              | string[pyarrow] | 551,443 non-null
 6. Genre                | string[pyarrow] | 551,443 non-null
 7. Album                | string[pyarrow] | 551,391 non-null
 8. Release Date         | string[pyarrow] | 551,443 non-null
 9. Key                  | string[pyarrow] | 551,443 non-null
10. Tempo                | int64[pyarrow] | 551,443 non-null
11. Loudness (db)        | string[pyarrow] | 551,443 non-null
12. Time signature       | string[pyarrow] | 551,435 non-null
13. Explicit             | string[pyarrow] | 551,443 non-null
14. Popularity           | int64[pyarrow] | 551,443 non-null
15. Energy               | 

,Artist(s),song,text,Length,emotion,Genre,Album,Release Date,Key,Tempo,Loudness (db),Time signature,Explicit,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Similar Artist 1,Similar Song 1,Similarity Score 1,Similar Artist 2,Similar Song 2,Similarity Score 2,Similar Artist 3,Similar Song 3,Similarity Score 3
0,!!!,Even When the Waters Cold,Friends told her she was better off at the bot...,03:47,sadness,hip hop,Thr!!!er,29th April 2013,D min,105,-6.85db,4/4,No,40,83,71,87,4,16,11,0,0,0,0,0,0,0,0,0,0,Corey Smith,If I Could Do It Again,0.986061,Toby Keith,Drinks After Work,0.983719,Space,Neighbourhood,0.983236
1,!!!,One Girl / One Boy,"Well I heard it, playing soft From a drunken b...",04:03,sadness,hip hop,Thr!!!er,29th April 2013,A# min,117,-5.75db,4/4,No,42,85,70,87,4,32,0,0,0,0,0,0,0,0,0,0,0,Hiroyuki Sawano,BRE@TH//LESS,0.995409,When In Rome,Heaven Knows,0.990905,Justice Crew,Everybody,0.984483
2,!!!,Pardon My Freedom,"Oh my god, did I just say that out loud? Shoul...",05:51,joy,hip hop,Louden Up Now,8th June 2004,A Maj,121,-6.06db,4/4,No,29,89,71,63,8,64,0,20,0,0,0,1,0,0,0,0,0,Ricky Dillard,More Abundantly Medley Live,0.993176,Juliet,Avalon,0.965147,The Jacksons,Lovely One,0.956752



=== Potential Matching Columns ===
No exact column matches found.
Main dataset columns: ['user_id', 'trackname', 'artistname', 'playlistname']
Metadata columns with 'name' or 'artist': ['Similar Artist 1', 'Similar Song 2', 'song', 'Artist(s)', 'Similar Song 3', 'Similar Song 1', 'Similar Artist 2', 'Similar Artist 3']


In [9]:
# Text preprocessing for better string matching between datasets
import re
import unicodedata

def clean_text_for_matching(text):
    """
    Comprehensive text cleaning for better string matching
    Best practices for music data joins:
    """
    if pd.isna(text) or text == '':
        return ''
    
    # Convert to string and lowercase
    text = str(text).lower().strip()
    
    # Remove Unicode accents/diacritics (café -> cafe)
    text = unicodedata.normalize('NFD', text)
    text = ''.join(char for char in text if unicodedata.category(char) != 'Mn')
    
    # Normalize numbers - convert written numbers to digits
    # Common in music: "twenty one pilots" vs "21 pilots"
    number_map = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
        'ten': '10', 'eleven': '11', 'twelve': '12', 'thirteen': '13', 
        'fourteen': '14', 'fifteen': '15', 'sixteen': '16', 'seventeen': '17',
        'eighteen': '18', 'nineteen': '19', 'twenty': '20', 'thirty': '30',
        'forty': '40', 'fifty': '50', 'sixty': '60', 'seventy': '70',
        'eighty': '80', 'ninety': '90', 'hundred': '100'
    }
    
    # Handle compound numbers like "twenty-one", "twenty one"
    text = re.sub(r'twenty[-\s]?one', '21', text)
    text = re.sub(r'twenty[-\s]?two', '22', text)
    text = re.sub(r'twenty[-\s]?three', '23', text)
    text = re.sub(r'twenty[-\s]?four', '24', text)
    text = re.sub(r'twenty[-\s]?five', '25', text)
    text = re.sub(r'twenty[-\s]?six', '26', text)
    text = re.sub(r'twenty[-\s]?seven', '27', text)
    text = re.sub(r'twenty[-\s]?eight', '28', text)
    text = re.sub(r'twenty[-\s]?nine', '29', text)
    
    # Replace simple number words
    for word_num, digit in number_map.items():
        text = re.sub(r'\b' + word_num + r'\b', digit, text)
    
    # Remove common music-specific noise
    # Remove "feat.", "ft.", "featuring", etc.
    text = re.sub(r'\b(feat\.?|ft\.?|featuring|with|vs\.?|versus)\b.*$', '', text)
    
    # Remove content in parentheses/brackets (remixes, versions, etc.)
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    
    # Remove common title suffixes
    text = re.sub(r'\b(remix|version|edit|mix|radio|explicit|clean|remaster|deluxe)\b', '', text)
    
    # Handle dashes and hyphens carefully (common in music)
    # Convert various dash types to standard space
    text = re.sub(r'[-–—]', ' ', text)  # Regular dash, en-dash, em-dash
    
    # Replace common symbols and punctuation
    text = re.sub(r'[&+]', 'and', text)  # & or + -> and
    text = re.sub(r'[.]', '', text)  # Remove periods (e.g., "feat." already handled)
    text = re.sub(r'[^\w\s]', '', text)  # Remove remaining punctuation except spaces
    
    # Handle multiple spaces and trim
    text = ' '.join(text.split())
    
    return text

def analyze_matching_potential(df_main, df_meta, main_artist_col, main_track_col, 
                              meta_artist_col, meta_track_col):
    """Analyze how well the datasets might match after cleaning"""
    
    print("=== STRING MATCHING ANALYSIS ===")
    
    # Sample data for analysis
    sample_size = min(5000, len(df_main), len(df_meta))
    main_sample = df_main.sample(sample_size)
    meta_sample = df_meta.sample(sample_size)
    
    print(f"Analyzing {sample_size:,} samples from each dataset...")
    
    # Clean the text columns
    print("\n1. Cleaning artist names...")
    main_artists_clean = main_sample[main_artist_col].apply(clean_text_for_matching)
    meta_artists_clean = meta_sample[meta_artist_col].apply(clean_text_for_matching)
    
    print("2. Cleaning track names...")
    main_tracks_clean = main_sample[main_track_col].apply(clean_text_for_matching)
    meta_tracks_clean = meta_sample[meta_track_col].apply(clean_text_for_matching)
    
    # Analyze overlaps
    print("\n=== OVERLAP ANALYSIS ===")
    
    # Artist overlap
    main_artist_set = set(main_artists_clean.dropna())
    meta_artist_set = set(meta_artists_clean.dropna())
    artist_overlap = len(main_artist_set.intersection(meta_artist_set))
    
    print(f"Artist name overlap: {artist_overlap:,} matches")
    print(f"  Main dataset unique artists: {len(main_artist_set):,}")
    print(f"  Metadata unique artists: {len(meta_artist_set):,}")
    print(f"  Overlap rate: {artist_overlap/len(main_artist_set)*100:.1f}%")
    
    # Track overlap  
    main_track_set = set(main_tracks_clean.dropna())
    meta_track_set = set(meta_tracks_clean.dropna())
    track_overlap = len(main_track_set.intersection(meta_track_set))
    
    print(f"\nTrack name overlap: {track_overlap:,} matches")
    print(f"  Main dataset unique tracks: {len(main_track_set):,}")
    print(f"  Metadata unique tracks: {len(meta_track_set):,}")
    print(f"  Overlap rate: {track_overlap/len(main_track_set)*100:.1f}%")
    
    # Show some examples
    print(f"\n=== EXAMPLE TRANSFORMATIONS ===")
    examples = main_sample.head(3)
    for i, row in examples.iterrows():
        artist_orig = row[main_artist_col]
        artist_clean = clean_text_for_matching(artist_orig)
        track_orig = row[main_track_col]
        track_clean = clean_text_for_matching(track_orig)
        
        print(f"\nExample {i+1}:")
        print(f"  Artist: '{artist_orig}' -> '{artist_clean}'")
        print(f"  Track:  '{track_orig}' -> '{track_clean}'")
    
    return {
        'artist_overlap': artist_overlap,
        'track_overlap': track_overlap,
        'main_artists': len(main_artist_set),
        'meta_artists': len(meta_artist_set),
        'main_tracks': len(main_track_set),
        'meta_tracks': len(meta_track_set)
    }

# Analyze matching potential between our datasets
if df is not None and df_metadata_full is not None:
    print("\n🔍 Analyzing string matching potential between datasets...")
    
    matching_stats = analyze_matching_potential(
        df, df_metadata_full,
        'artistname', 'trackname',
        'Artist(s)', 'song'
    )
    
    print(f"\n💡 RECOMMENDATIONS:")
    if matching_stats['artist_overlap'] > 100:
        print("✅ Good artist name overlap - merge should work well")
    else:
        print("⚠️  Low artist overlap - consider fuzzy matching")
        
    if matching_stats['track_overlap'] > 100:
        print("✅ Good track name overlap - merge should work well")
    else:
        print("⚠️  Low track overlap - consider alternative strategies")
        
else:
    print("❌ Need both datasets loaded to analyze matching potential")


🔍 Analyzing string matching potential between datasets...
=== STRING MATCHING ANALYSIS ===
Analyzing 5,000 samples from each dataset...

1. Cleaning artist names...
Analyzing 5,000 samples from each dataset...

1. Cleaning artist names...
2. Cleaning track names...
2. Cleaning track names...

=== OVERLAP ANALYSIS ===
Artist name overlap: 604 matches
  Main dataset unique artists: 3,304
  Metadata unique artists: 4,108
  Overlap rate: 18.3%

Track name overlap: 243 matches
  Main dataset unique tracks: 4,772
  Metadata unique tracks: 4,733
  Overlap rate: 5.1%

=== EXAMPLE TRANSFORMATIONS ===

Example 7477634:
  Artist: 'The Prodigy' -> 'the prodigy'
  Track:  'Your Love (Remix) (Remastered)' -> 'your love'

Example 2670826:
  Artist: 'Astronautalis' -> 'astronautalis'
  Track:  'The Wondersmith and His Sons' -> 'the wondersmith and his sons'

Example 3114316:
  Artist: 'Taylor Swift' -> 'taylor swift'
  Track:  'I Knew You Were Trouble' -> 'i knew you were trouble'

💡 RECOMMENDATIONS:

In [ ]:
# Test the text preprocessing function (optional - can be commented out)
print("🔧 TESTING COMPREHENSIVE TEXT NORMALIZATION:")
test_cases = [
    # Number normalization
    "Twenty One Pilots",
    "21 Pilots", 
    "Maroon 5",
    "Maroon Five",
    "twenty-three",
    "23",
    
    # Dash and hyphen handling
    "Jay-Z",
    "Jay Z",
    "X-Men",
    "X Men", 
    "Three Days Grace",
    "Three-Days-Grace",
    
    # Special characters and punctuation
    "P!nk",
    "Pink",
    "Panic! At The Disco",
    "Panic At The Disco",
    "AC/DC",
    "ACDC",
    "Guns N' Roses",
    "Guns N Roses",
    
    # Unicode and accents
    "Café Tacuba",
    "Cafe Tacuba",
    "Björk",
    "Bjork",
    "Motörhead", 
    "Motorhead",
    
    # Featuring and collaboration indicators
    "Song Title (feat. Artist)",
    "Song Title feat Artist",
    "Artist & Another Artist",
    "Artist and Another Artist",
    
    # Mixed cases
    "twenty-one pilots",
    "TWENTY ONE PILOTS",
    "Twenty-One Pilots"
]

print("Before → After normalization:")
for case in test_cases:
    cleaned = clean_text_for_matching(case)
    print(f"  '{case}' → '{cleaned}'")

## 4. Advanced Visualizations and Analysis

This section will contain advanced analysis techniques - add your specific analysis here

In [ ]:
# Placeholder for advanced analysis
# This cell is ready for you to add specific analysis based on your research questions

print("Ready to add advanced analysis!")
print("Suggested analysis directions:")
print("1. User clustering based on listening patterns")
print("2. Artist similarity networks")
print("3. Temporal analysis of playlist creation")
print("4. Recommendation system prototyping")
print("5. Integration with Brandon's dataset processing")

## 5. Research Questions Analysis

Address the specific research questions from the project proposal

In [ ]:
# Framework for answering research questions:
# Q1: Can we identify distinct groups of artists who are enjoyed by similar listeners?
# Q2: Can we predict which songs a user is likely to add to their playlist?
# Q3: Can we predict playlist additions based on song elements?

print("Research Questions Analysis Framework:")
print("\nQ1: Artist Groups Analysis")
print("- Build artist co-occurrence networks")
print("- Cluster artists by shared listeners")
print("- Analyze genre/style similarities")

print("\nQ2: User Preference Prediction")
print("- Collaborative filtering approach")
print("- User similarity based on listening history")
print("- Matrix factorization techniques")

print("\nQ3: Content-Based Prediction")
print("- Integration with music metadata")
print("- Audio feature analysis")
print("- Playlist context modeling")

## 6. Next Steps and Integration

Plan for connecting with team analysis and model development

In [ ]:
# Summary and next steps
print("=== EDA Part 2 - Next Steps ===")
print("\n1. Team Integration:")
print("   - Connect with Brandon's data_modeling.ipynb")
print("   - Use insights from dataset.ipynb for data preparation")
print("   - Coordinate feature engineering approaches")

print("\n2. Model Development:")
print("   - Implement collaborative filtering")
print("   - Develop content-based recommendations")
print("   - Hybrid model combining both approaches")

print("\n3. Evaluation:")
print("   - Define success metrics")
print("   - Cross-validation strategies")
print("   - User study design")

print("\n4. Visualization:")
print("   - Interactive dashboards")
print("   - Network visualizations")
print("   - Recommendation explanation interfaces")